<a href="https://colab.research.google.com/github/Yuktha-J/AI_LAB/blob/main/1BM24CS339_Week8_ForwardChaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def split_parts(s):
    parts = []
    start = 0
    level = 0
    for i in range(len(s)):
        if s[i] == '(':
            level += 1
        elif s[i] == ')':
            level -= 1
        elif s[i] == ',' and level == 0:
            parts.append(s[start:i].strip())
            start = i + 1
    parts.append(s[start:].strip())
    return parts

def parse(expression):
    expression = expression.strip()
    if '(' not in expression:
        return expression, []
    name = expression[:expression.index('(')].strip()
    inside = expression[
        expression.index('(') + 1:
        expression.rindex(')')
    ]
    arguments = split_parts(inside)
    return name, arguments

def is_variable(term):
    return (
        len(term) > 0
        and term[0].islower()
        and '(' not in term
    )

def substitute(expression, substitution):
    name, arguments = parse(expression)
    if len(arguments) == 0:
        if name in substitution:
            return substitute(
                substitution[name],
                substitution
            )
        return name
    new_arguments = []
    for argument in arguments:
        new_arguments.append(
            substitute(argument, substitution)
        )
    return name + "(" + ",".join(new_arguments) + ")"

def unify(x, y, substitution):
    x = substitute(x, substitution)
    y = substitute(y, substitution)
    if x == y:
        return substitution
    if is_variable(x):
        substitution[x] = y
        return substitution
    if is_variable(y):
        substitution[y] = x
        return substitution
    name_x, args_x = parse(x)
    name_y, args_y = parse(y)
    if name_x != name_y:
        return None
    if len(args_x) != len(args_y):
        return None
    for a, b in zip(args_x, args_y):
        result = unify(a, b, substitution)
        if result is None:
            return None
    return substitution

def find_matches(conditions, facts, substitution):
    if len(conditions) == 0:
        return [substitution]
    condition = substitute(
        conditions[0],
        substitution
    )
    results = []
    for fact in facts:
        new_substitution = substitution.copy()
        result = unify(
            condition,
            fact,
            new_substitution
        )
        if result is not None:
            remaining_conditions = conditions[1:]
            matches = find_matches(
                remaining_conditions,
                facts,
                result
            )
            results.extend(matches)
    return results

def forward_reasoning(facts, rules, query):
    while True:
        new_facts = []
        for conditions, conclusion in rules:
            matches = find_matches(
                conditions,
                facts,
                {}
            )
            for substitution in matches:
                new_fact = substitute(
                    conclusion,
                    substitution
                )
                if (
                    new_fact not in facts
                    and new_fact not in new_facts
                ):
                    new_facts.append(new_fact)
                    result = unify(
                        query,
                        new_fact,
                        {}
                    )
                    if result is not None:
                        return True, new_fact
        if len(new_facts) == 0:
            return False, None
        facts.extend(new_facts)

print("FORWARD REASONING ALGORITHM")
print("---------------------------")
n = int(input("Enter number of facts: "))
facts = []
print("Enter facts:")
for i in range(n):
    facts.append(input().strip())
r = int(input("Enter number of rules: "))
rules = []
print("Enter rules:")
for i in range(r):
    rule = input().strip()
    left, right = rule.split("->")
    conditions = split_parts(left.strip())
    conclusion = right.strip()
    rules.append(
        (conditions, conclusion)
    )
query = input("Enter query: ").strip()
result, new_fact = forward_reasoning(
    facts,
    rules,
    query
)
print()
if result:
    print("Query is TRUE")
    print("Derived:", new_fact)
else:
    print("Query is FALSE")

FORWARD REASONING ALGORITHM
---------------------------
Enter number of facts: 4
Enter facts:
Owns(A,T1)
Missile(T1)
American(Robert)
Enemy(A,America)
Enter number of rules: 4
Enter rules:
Missile(x) -> Weapon(x)
Missile(x), Owns(A,x) -> Sells(Robert,x,A)
Enemy(x,America) -> Hostile(x)
American(x), Weapon(y), Sells(x,y,z), Hostile(z) -> Criminal(x)
Enter query: Criminal(Robert)

Query is TRUE
Derived: Criminal(Robert)
